In [14]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from meteor import MeteorInterface

In [2]:
# FASTMIP Phase 1 ESMs:
ESMs = ['ACCESS-ESM1-5']#, 'CMCC-CM2-SR5', 'CNRM-CM6-1', 'CanESM5', 'EC-Earth3', 'INM-CM5-0', 'IPSL-CM6A-LR', 'MIROC-ES2L', 'MIROC6', 'MPI-ESM1-2-LR', 'MPI-ESM1-2-HR', 'MRI-ESM2-0']

# FASTMIP Phase 2 ESMs:
ESMs_Tier1 = ['ACCESS-ESM1-5', 'CanESM5', 'IPSL-CM6A-LR', 'MPI-ESM1-2-LR', 'MIROC6']
ESMs_Tier2 = ['ACCESS-CM2', 'AWI-CM-1-1-MR', 'BCC-CSM2-MR', 'CanESM5-1', 'CESM2', 'CESM2-WACCM', 'CNRM-CM6-1', 'CNRM-ESM2-1', 'EC-Earth3', 'FGOALS-g3', 'FIO-ESM-2-0', 'GFDL-ESM4', 'GISS-E2-1-G', 'HadGEM3–GC31-LL', 'HadGEM3–GC31-MM', 'MIROC-ES2L', 'MRI-ESM2-0', 'NorESM2-LM', 'UKESM1-0-LL', 'MPI-ESM1-2-HR']

In [3]:
# FASTMIP Phase 1 output to generate:
# Up to 2100
# 2.5x2.5 degree common grid
# 10 member ensemble for each ESM and scenario
# Annual mean tas (gridded)
# Monthly mean tas and pr (gridded)
# Both METEOR raw output and scaled to provided GSAT timeseries
# output filename convention for FASTMIP phase 1: emulations_{tas, pr}_{ann, mon}_METEOR_{esm}_{scenario}_g025.nc 

scenarios = ['ssp119']#, 'ssp126', 'ssp245', 'ssp370', 'ssp585']
FAIRpercentiles = ['5', '10', '50', '90', '95']
start_year = 2020
end_year = 2100
n_members = 10
output_dir = '../data/FASTMIP_phase1/METEOR_emulations/'
scaling_dir = '../data/FASTMIP_phase1/FAIR_GSAT/'

def format_FAIRssp(scenario):
    return f"SSP{scenario[3]}-{scenario[4:]}"

In [11]:
xr.open_dataset(f"{scaling_dir}GSAT_SSPmarker_SSP1-19.nc")['tas'].sel(percentile='50', SCM='FaIRv1.6.2')

<xarray.DataArray 'tas' (time: 86, IAM: 1)> Size: 688B
[86 values with dtype=float64]
Coordinates:
  * time        (time) int64 688B 2015 2016 2017 2018 ... 2097 2098 2099 2100
  * IAM         (IAM) <U11 44B 'IMAGE 3.0.1'
    percentile  <U2 8B '50'
    SCM         <U12 48B 'FaIRv1.6.2'
Attributes:
    long_name:   Air Temperature
    short_name:  tas
    units:       K
    type:        unstructured
    reference:   1850-1900

In [16]:
pd.read_csv('../data/FASTMIP_phase2/climate_assessment_forced.csv')

,model,scenario,region,variable,unit,ensemble_member,climate_model,calibration,1850,1851,...,2092,2093,2094,2095,2096,2097,2098,2099,2100,2101
0,AIM 3.0,Low-to-Negative - SSP2 (Marker),World,Climate Assessment|Surface Temperature (GSAT),K,385,fair-2.2.4,1.6.0-forced,0.123227,0.138781,...,1.449714,1.448139,1.445634,1.444386,1.437469,1.421611,1.409126,1.399909,1.391798,1.388131
1,AIM 3.0,Low-to-Negative - SSP2 (Marker),World,Climate Assessment|Surface Temperature (GSAT),K,1926,fair-2.2.4,1.6.0-forced,0.105229,0.125141,...,1.589544,1.584149,1.578033,1.572178,1.563924,1.548895,1.531036,1.514579,1.499878,1.488253
2,AIM 3.0,Low-to-Negative - SSP2 (Marker),World,Climate Assessment|Surface Temperature (GSAT),K,2551,fair-2.2.4,1.6.0-forced,0.111275,0.130248,...,1.958441,1.950476,1.942074,1.934780,1.923810,1.907287,1.893960,1.882623,1.871471,1.862830
3,AIM 3.0,Low-to-Negative - SSP2 (Marker),World,Climate Assessment|Surface Temperature (GSAT),K,9084,fair-2.2.4,1.6.0-forced,-0.072371,-0.047156,...,2.526700,2.520739,2.513510,2.506868,2.496341,2.478920,2.462872,2.449188,2.436298,2.426199
4,AIM 3.0,Low-to-Negative - SSP2 (Marker),World,Climate Assessment|Surface Temperature (GSAT),K,9125,fair-2.2.4,1.6.0-forced,0.122412,0.140613,...,1.530170,1.523672,1.516488,1.510456,1.500472,1.484061,1.470578,1.459564,1.449062,1.441460
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30271,WITCH 6.0,High-to-Low - SSP5 (Marker),World,Climate Assessment|Concentration|CO2,ppm,1595155,fair-2.2.4,1.6.0-forced,285.376107,285.513008,...,563.176013,562.399340,561.585665,560.732448,559.838093,558.922347,557.981603,557.014694,556.020846,554.999126
30272,WITCH 6.0,High-to-Low - SSP5 (Marker),World,Climate Assessment|Concentration|CO2,ppm,1595160,fair-2.2.4,1.6.0-forced,283.313327,283.469525,...,522.682620,521.647092,520.595637,519.525383,518.434487,517.342441,516.245571,515.141540,514.028723,512.905862
30273,WITCH 6.0,High-to-Low - SSP5 (Marker),World,Climate Assessment|Concentration|CO2,ppm,1595249,fair-2.2.4,1.6.0-forced,283.854012,284.008738,...,539.744576,538.775744,537.780894,536.756590,535.701111,534.633712,533.549523,532.446806,531.325389,530.184944
30274,WITCH 6.0,High-to-Low - SSP5 (Marker),World,Climate Assessment|Concentration|CO2,ppm,1597667,fair-2.2.4,1.6.0-forced,280.828194,280.985869,...,555.538848,554.609092,553.644260,552.641639,551.599944,550.538110,549.450904,548.337332,547.197276,546.030362


In [10]:
# Create emulators for list of ESMs (uses cached models if available)
for esm in ESMs:
    print(f"Creating emulator for {esm}...")
    emulator = MeteorInterface(
        model=esm,
        variables=['tas', 'pr'],
        cache_dir='../cache'
    )
    print(f"Emulator created for {emulator.model} with variables: {emulator.variables}")
    
    print(f"Training emulator for {emulator.model}...")
    emulator.train(verbose=False)

    for scenario in scenarios:
        print(f"Emulating scenario {scenario}...")

        # Emulate gridded outputs for each scenario and save:
        ensemble_gridded = emulator.generate_ensemble_outputs(
            scenario=scenario,
            start_year=start_year,
            end_year=end_year,
            n_realizations=n_members,
            timeseries=['global'], 
            gridded={
                'annual': list(range(start_year, end_year + 1)),  # Annual mean grids for all years 
                'monthly': list(range(start_year, end_year + 1)),  # Monthly mean grids for all years
            },
            save_to=f"{output_dir}METEOR_{esm}_{scenario}.nc"
        )

        # Scale output to FAIR GSAT:
        # (Could also loop through percentiles here, for now only using 50th)
        for percentile in ['50']: #FAIRpercentiles:    
            scaling_ts = xr.open_dataset(f"{scaling_dir}GSAT_SSPmarker_{format_FAIRssp(scenario)}.nc")['tas'].sel(percentile=percentile, SCM='FaIRv1.6.2', time=slice(start_year, end_year)).rename({'time': 'year'})

            ensemble_scaled = emulator.generate_ensemble_outputs(
                scenario=scenario,
                start_year=start_year,
                end_year=end_year,
                n_realizations=n_members,
                timeseries=['global'], 
                gridded={
                    'annual': list(range(start_year, end_year + 1)),  # Annual mean grids for all years
                    'monthly': list(range(start_year, end_year + 1)),  # Monthly mean grids for all years 
                },
                save_to=f"{output_dir}METEOR_{esm}_{scenario}_scaledtoFAIR_{percentile}percentile.nc",
                temp_scaling_ts=scaling_ts
            )   
        

Creating emulator for ACCESS-ESM1-5...
Emulator created for ACCESS-ESM1-5 with variables: ['tas', 'pr']
Training emulator for ACCESS-ESM1-5...
🔧 Preparing pattern scaling training data for ACCESS-ESM1-5...
   ✅ Training data prepared for experiments:  ['base', 'co2x4', 'ssp245', 'sulxanom']
📥 Loading CICERO-SCM forcing data for ssp245...
   ✅ Loaded 801 concentration records
   ✅ Loaded 351 emission records
   ✅ Config: 1750-2100, emissions start: 1850
📦 Loading cached pattern scaling model from ../cache/pattern_scaling/cmip6-ACCESS-ESM1-5-aer-tas_pattern_scaling.pkl
✅ Pattern scaling model loaded from ../cache/pattern_scaling/cmip6-ACCESS-ESM1-5-aer-tas_pattern_scaling.pkl
Model loaded from ../cache/noise_models/ACCESS-ESM1-5_tas_noise_model.pkl
🔧 Preparing pattern scaling training data for ACCESS-ESM1-5...
   ✅ Training data prepared for experiments:  ['base', 'co2x4', 'ssp245', 'sulxanom']
📥 Loading CICERO-SCM forcing data for ssp245...
   ✅ Loaded 801 concentration records
   ✅ Loa

ValueError: temp_scaling_ts temporal extent (81) must match annual_prediction time dimension (351)

In [5]:
xr.open_dataset(f"{output_dir}METEOR_ACCESS-ESM1-5_ssp119.nc")

<xarray.Dataset> Size: 5GB
Dimensions:           (year: 81, realization: 10, lat: 145, lon: 192, date: 972)
Dimensions without coordinates: year, realization, lat, lon, date
Data variables:
    tas_grid_annual   (year, realization, lat, lon) float64 180MB ...
    tas_grid_monthly  (date, realization, lat, lon) float64 2GB ...
    tas_global        (realization, date) float64 78kB ...
    pr_grid_annual    (year, realization, lat, lon) float64 180MB ...
    pr_grid_monthly   (date, realization, lat, lon) float64 2GB ...
    pr_global         (realization, date) float64 78kB ...
Attributes:
    model:           ACCESS-ESM1-5
    scenario:        ssp119
    year_range:      2020-2100
    n_realizations:  10

In [5]:
def test_to_netcdf(ensemble_gridded, output_path):
    datasets = {}

    
    for var_name, var_data in ensemble_gridded.variables.items():
        ds = xr.Dataset()
        print(var_name)

        # Do gridded variables first, because these actually return the years as coordinates, whereas the timeseries variables just return arrays without start/end year information
   
        for grid_name, grid_array in var_data.gridded.items():
            print(grid_name)
            safe_name = str(grid_name).replace("-", "_").replace(":", "_")
            if isinstance(grid_array, dict) and grid_name == "annual":
                    years = sorted(grid_array.keys())
                    stacked = xr.concat(
                        [grid_array[y] for y in years],
                        dim="year"
                    )
                    stacked = stacked.assign_coords(year=years)
                    ds[f"{var_name}_grid_{safe_name}"] = stacked

            elif isinstance(grid_array, dict) and grid_name == "monthly":
                    years = sorted(grid_array.keys())
                    stacked = xr.concat(
                                [grid_array[y] for y in years],
                                dim="year").assign_coords(year=years)
                    stacked = stacked.stack(date=("year", "month"))
                    date_vals = [y * 100 + m
                                 for y in years
                                 for m in range(1, 13)]
                    stacked = stacked.drop_vars(['date', 'year', 'month']).assign_coords(date=date_vals).transpose("date", "realization", "lat", "lon")
                    ds[f"{var_name}_grid_{safe_name}"] = stacked
    

        for ts_name, ts_array in var_data.timeseries.items():
            safe_name = ts_name.replace(":", "_").replace(".", "p")
            if ts_array.ndim == 1:
                dims = ("date",)
            elif ts_array.ndim == 2:
                dims = ("realization", "date")
            ds[f"{var_name}_{safe_name}"] = (dims, ts_array)
     
        datasets[var_name] = ds

    combined = xr.merge(list(datasets.values()))
    return combined

In [7]:
grid_array=ensemble_gridded['tas'].gridded['monthly']
years = sorted(grid_array.keys())
stacked = xr.concat(
            [grid_array[y] for y in years],
            dim="year"
        ).assign_coords(year=years)
stacked = stacked.stack(date=("year", "month"))
date_vals = [
                y * 100 + m
                for y in years
                for m in range(1, 13)
            ]
stacked = stacked.drop_vars(['date', 'year', 'month']).assign_coords(date=date_vals)

In [6]:
ds = test_to_netcdf(ensemble_gridded, "test_output.nc")

tas
annual
monthly
pr
annual
monthly


In [7]:
ds

<xarray.Dataset> Size: 5GB
Dimensions:           (lat: 145, lon: 192, year: 81, realization: 10, date: 972)
Coordinates:
  * lat               (lat) float64 1kB -90.0 -88.75 -87.5 ... 87.5 88.75 90.0
  * lon               (lon) float64 2kB 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * year              (year) int64 648B 2020 2021 2022 2023 ... 2098 2099 2100
  * date              (date) int64 8kB 202001 202002 202003 ... 210011 210012
Dimensions without coordinates: realization
Data variables:
    tas_grid_annual   (year, realization, lat, lon) float64 180MB 0.724 ... 5...
    tas_grid_monthly  (date, realization, lat, lon) float64 2GB 21.25 ... -4.47
    tas_global        (realization, date) float64 78kB -0.5665 ... 0.1769
    pr_grid_annual    (year, realization, lat, lon) float64 180MB 3.015e-07 ....
    pr_grid_monthly   (date, realization, lat, lon) float64 2GB -1.51e-07 ......
    pr_global         (realization, date) float64 78kB 3.753e-05 ... 3.837e-05

In [15]:
ds

<xarray.Dataset> Size: 78kB
Dimensions:    (realization: 10, month: 972)
Dimensions without coordinates: realization, month
Data variables:
    pr_global  (realization, month) float64 78kB 3.753e-05 ... 3.844e-05

In [29]:
ensemble_gridded['tas'].gridded['monthly'][2020]

<xarray.DataArray (realization: 10, month: 12, lat: 145, lon: 192)> Size: 27MB
array([[[[ 2.12450928e+01,  2.12450928e+01,  2.12450928e+01, ...,
           2.12450928e+01,  2.12450928e+01,  2.12450928e+01],
         [ 1.92392010e+01,  1.92483822e+01,  1.92571206e+01, ...,
           1.92086089e+01,  1.92200063e+01,  1.92297519e+01],
         [ 1.91607723e+01,  1.91818512e+01,  1.91995545e+01, ...,
           1.90831996e+01,  1.91113179e+01,  1.91374919e+01],
         ...,
         [-8.16713841e+00, -8.13483657e+00, -8.10231279e+00, ...,
          -8.27229917e+00, -8.23422455e+00, -8.20159989e+00],
         [-8.51343033e+00, -8.49967613e+00, -8.48679711e+00, ...,
          -8.55785368e+00, -8.54505144e+00, -8.52920224e+00],
         [-8.92147454e+00, -8.92147454e+00, -8.92147454e+00, ...,
          -8.92147454e+00, -8.92147454e+00, -8.92147454e+00]],

        [[ 1.18380076e+01,  1.18380076e+01,  1.18380076e+01, ...,
           1.18380076e+01,  1.18380076e+01,  1.18380076e+01],
         [ 1.07544613e+01,  1.07593652e+01,  1.07638266e+01, ...,
           1.07393058e+01,  1.07450375e+01,  1.07497949e+01],
         [ 1.07321041e+01,  1.07409230e+01,  1.07478854e+01, ...,
           1.06964850e+01,  1.07100056e+01,  1.07220751e+01],
...
          -5.64694077e-01, -5.06042477e-01, -4.61094896e-01],
         [ 5.91163340e-02,  8.62369363e-02,  1.13608049e-01, ...,
          -1.87001367e-02,  1.86156556e-03,  3.00965467e-02],
         [ 5.37913711e-01,  5.37913711e-01,  5.37913711e-01, ...,
           5.37913711e-01,  5.37913711e-01,  5.37913711e-01]],

        [[ 2.21909368e+01,  2.21909368e+01,  2.21909368e+01, ...,
           2.21909368e+01,  2.21909368e+01,  2.21909368e+01],
         [ 2.02544070e+01,  2.02484939e+01,  2.02417926e+01, ...,
           2.02649366e+01,  2.02632849e+01,  2.02590576e+01],
         [ 2.04836691e+01,  2.04713853e+01,  2.04518187e+01, ...,
           2.04825546e+01,  2.04891558e+01,  2.04904919e+01],
         ...,
         [-4.53434201e+00, -4.52761123e+00, -4.52139274e+00, ...,
          -4.56096207e+00, -4.54489070e+00, -4.54138283e+00],
         [-4.33821211e+00, -4.33658934e+00, -4.33669698e+00, ...,
          -4.34156135e+00, -4.34115518e+00, -4.34085661e+00],
         [-4.25378900e+00, -4.25378900e+00, -4.25378900e+00, ...,
          -4.25378900e+00, -4.25378900e+00, -4.25378900e+00]]]],
      shape=(10, 12, 145, 192))
Coordinates:
  * month    (month) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
  * lat      (lat) float64 1kB -90.0 -88.75 -87.5 -86.25 ... 87.5 88.75 90.0
  * lon      (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
Dimensions without coordinates: realization